[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter11_the_training_loop/notebook.ipynb)

# Chapter 11: The Training Loop

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

In the summer of 1935, a botanist named Edgar Anderson was working on the Gaspé Peninsula in Quebec, trying to understand how closely related species of iris differ from one another. He found two species growing together and measured them carefully, taking the length and width of the sepal and the petal of each flower. What made his measurements unusually good was the discipline behind them. The flowers were all from the same pasture, picked on the same day, and measured at the same time by the same person with the same apparatus, so almost nothing varied except the flowers themselves.

The next year the statistician Ronald Fisher took Anderson's measurements, added an earlier set for a third species, and published a paper using the 150 flowers to show how several measurements together could separate groups that any single measurement would confuse. Those 150 flowers have been used ever since. They are small enough to print, old enough to be free of any licensing question, and hard enough to be interesting, because one of the three species is easy to tell apart and the other two overlap.

Every network in this book so far has learned from numbers you generated yourself. This chapter trains one on Anderson's flowers, which is the first real data in this book, and in doing so it assembles everything from Part 2 and Part 3 into the loop you will use for the rest of your deep learning life. Before you read on, try to guess the answers to these four questions.

1. **What are the steps a network repeats over and over while it learns?**
2. **Does the order of those steps matter?**
3. **With three species rather than two, how many numbers should the network output?**
4. **If the training loss keeps falling but the validation loss turns upwards, what is happening?**

## The Five Steps of Training

Every training loop in PyTorch, from this one to a model with billions of weights, is the same five steps repeated. You have already done all of them by hand.

1. **Forward.** Run the data through the model to get its predictions, as in Chapter 3.
2. **Loss.** Score those predictions against the right answers, as in Chapter 5.
3. **Clear.** Wipe the old gradients, the chore Chapter 9 warned about.
4. **Backward.** Work out every gradient, as in Chapter 7.
5. **Step.** Move every weight a little downhill, as in Chapter 6.

In code those are five lines, and Figure 11.1 shows them as the cycle they really are.

> 🖼️ **Figure 11.1:** The five steps, with the PyTorch call for each. Every training run in this book is this cycle, repeated.

Only one ordering rule really matters. The clearing must happen before the backward pass, never after it, because gradients add up rather than replace each other. Putting `zero_grad()` directly before `backward()` every time is a habit worth forming now. One full pass through the training data is called an **epoch**, which is the word you saw counting up in Chapter 7.

## Writing the Loop

Real data needs preparation before any of that can run. The next cell loads Anderson's flowers, which ship inside scikit learn so nothing is downloaded.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
print(iris.data.shape)
print(iris.feature_names)
print(iris.target_names)

One hundred and fifty flowers, four measurements each, three species with fifty flowers apiece. Figure 11.2 plots two of the four measurements and shows immediately what makes this a good test. One species sits completely apart, while the other two touch along a border, so a model has an easy part and a hard part.

> 🖼️ **Figure 11.2:** Two of the four measurements. Setosa is easy, while versicolor and virginica meet along a border, which is where any mistakes will come from.

Books 4 and 5 drilled two habits that matter here. Split the data before touching it, and work out any scaling from the training half only, so the test half stays genuinely unseen.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3,
    random_state=7, stratify=iris.target)
print(X_train.shape, X_test.shape)

The `stratify` argument keeps all three species equally represented in both halves. Scaling matters more for a network than it did in Book 4, because Chapter 6 showed that large inputs make large slopes, so a learning rate suiting one column may wreck another.

In [ ]:
import torch
import torch.nn as nn

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
print(mean.round(2), std.round(2))

Those two lines come from the training half alone and are then applied to both halves. Converting to tensors is where Chapter 8's warning about types earns its keep, because the measurements must be float32 while the species labels must be whole numbers.

In [ ]:
Xtr = torch.tensor((X_train - mean) / std, dtype=torch.float32)
Xte = torch.tensor((X_test - mean) / std, dtype=torch.float32)
ytr = torch.tensor(y_train)
yte = torch.tensor(y_test)
print(Xtr.dtype, ytr.dtype)

The labels are whole numbers because they are names rather than quantities, and there is no sense in which virginica is twice versicolor.

The model takes the four measurements in and gives three numbers out, one score per species, which answers the third opening question. The shape rule from Chapter 3 fixes both ends of the network, since the input must match the four columns and the output must match the three species. Only the middle is a free choice, and 16 is a reasonable first guess, large enough to bend the space as Chapter 4 described and small enough to train in a moment. Chapter 20 returns to choosing that width deliberately rather than by guess.

In [ ]:
torch.manual_seed(7)
model = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 3))
print(sum(p.numel() for p in model.parameters()))

There is no softmax on the end, and that catches almost everyone. PyTorch's `CrossEntropyLoss` applies the softmax from Chapter 5 itself, as part of computing the loss, for reasons of numerical safety. Applying it yourself first means it happens twice, which does not crash but quietly trains a worse model.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
scores = torch.tensor([[2.0, 1.0, 0.1]])
target = torch.tensor([0])
print(round(loss_fn(scores, target).item(), 4))
print(round(loss_fn(torch.softmax(scores, dim=1), target).item(), 4))

The first number is the 0.417 you computed by hand in Chapter 5 for exactly these three scores. The second, from helpfully applying softmax first, is nearly twice as large and simply wrong. **Give `CrossEntropyLoss` the raw scores straight from the last linear layer, because it applies softmax internally.**

With the optimizer from Chapter 10, the loop is the five steps.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

history = []
for epoch in range(301):
    pred = model(Xtr)
    loss = loss_fn(pred, ytr)
    history.append(loss.item())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

That is the complete trainer, and every line of it came from a chapter you have already worked through.

## Your First Trained Network, a Full Multi Class Classification Walkthrough

Watching the loss is the first check that anything happened at all.

In [ ]:
for epoch in [0, 50, 100, 200, 300]:
    print(epoch, round(history[epoch], 4))

Look at where it started. Chapter 5 said a network that has learned nothing about three equally likely classes should score the logarithm of 3, which is 1.0986, and the untrained model began at 1.174, just above that. By epoch 300 it is at 0.0987, more than ten times lower. The network learned.

A loss is not an answer, though, so the next cell turns scores into species. The highest of the three scores wins, which is what `argmax` finds.

In [ ]:
with torch.no_grad():
    train_acc = (model(Xtr).argmax(1) == ytr).float().mean()
    test_out = model(Xte)
    test_acc = (test_out.argmax(1) == yte).float().mean()
    test_loss = loss_fn(test_out, yte)
print(round(train_acc.item(), 3), round(test_acc.item(), 3))
print(round(test_loss.item(), 4))

Ninety six percent of the training flowers and ninety six percent of the held out flowers, with a test loss of 0.1245 against a training loss of 0.0987. Those two pairs being so close is the sign of a healthy model, because the network did roughly as well on flowers it had never seen as on the ones it studied.

Accuracy alone hides where a model struggles, so the next cell counts each species separately.

In [ ]:
predicted = test_out.argmax(1)
for c, name in enumerate(iris.target_names):
    mask = yte == c
    print(name, int((predicted[mask] == c).sum()), "of", int(mask.sum()))

Exactly as Figure 11.2 predicted. Setosa, the species that sits apart, is perfect, and the only two mistakes are on the pair that overlap. The last cell asks how confident the network was when it got those two wrong.

In [ ]:
for i in (predicted != yte).nonzero().flatten().tolist():
    probs = torch.softmax(test_out[i], 0)
    print(iris.target_names[yte[i]], "called",
          iris.target_names[predicted[i]],
          [round(v, 2) for v in probs.tolist()])

This is the best possible news about a mistake. On the first flower the network split 42 against 58, barely preferring the wrong answer, and on the second it was only moderately sure. Chapter 5 explained that cross entropy punishes confident errors far more harshly than hesitant ones, and this model is hesitating exactly where the two species genuinely overlap. It is wrong in the way a careful botanist would be wrong.

## Reading the Loss Curve

One number at the end of training tells you much less than the shape of the whole run. The honest way to watch is to record the loss on both halves every epoch, measuring the held out half inside `no_grad` since no learning happens there.

In [ ]:
torch.manual_seed(7)
model = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3))
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

train_curve, test_curve = [], []
for epoch in range(301):
    loss = loss_fn(model(Xtr), ytr)
    train_curve.append(loss.item())
    with torch.no_grad():
        test_curve.append(loss_fn(model(Xte), yte).item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(round(train_curve[-1], 4), round(test_curve[-1], 4))

Figure 11.3 draws both. They fall together and stay close, with the held out line sitting a little above the training line, which is normal and healthy.

> 🖼️ **Figure 11.3:** Both curves for the iris run. They fall together and the gap stays small, which is what a healthy run looks like. The dashed line is Chapter 5's baseline for three classes.

What makes the picture useful is that unhealthy runs look different, and each shape points at a different cause. Table 11.1 collects the ones you will actually meet.

**Table 11.1: Loss curve shapes and what each one is telling you. Reading this picture is the single most useful habit in training networks.**

| What the curves do | What it means | Where it is handled |
|---|---|---|
| Both fall, gap stays small | Healthy, as here | This chapter |
| Training falls, held out turns up | Overfitting, memorising the training flowers | Chapter 15 |
| Neither moves from the baseline | Not learning at all | Chapters 14 and 16 |
| Loss jumps about or becomes nan | Learning rate too large | Chapters 6 and 14 |
| Both flatten while still high | Model too small for the problem | Chapter 20 |

**Always record the loss on data the network is not learning from, because the training loss alone cannot tell the difference between a model that is learning and one that is memorising.**

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Skip the scaling.** Build `Xtr` and `Xte` from the raw measurements with no mean and spread adjustment, train again for 300 epochs and compare the final loss with 0.0987.
2. **Move the clearing.** Put `optimizer.zero_grad()` after `optimizer.step()` instead of before `backward()`, run again, and explain the result using the accumulation from Chapter 9.
3. **Shrink the model.** Replace the 16 hidden neurons with 2, train again, and look at which species it now confuses and how confident it is when it is wrong.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **What are the five steps of a training loop, in order?**
2. **Why must the gradients be cleared before the backward pass?**
3. **Why does the model have no softmax on its output?**
4. **Why was the mean and spread computed from the training half only?**
5. **What does it mean when the training loss falls but the held out loss rises?**

### Answers

1. Forward, loss, clear the gradients, backward, step.
2. Because gradients add up instead of replacing each other, so leftovers from the previous epoch would be added to this epoch's.
3. Because `CrossEntropyLoss` applies softmax itself, so doing it first would apply it twice.
4. Because the test half must stay genuinely unseen, and using it to compute the scaling would leak information about it into training.
5. That the model is overfitting, learning the training examples themselves rather than the pattern behind them.

## What Comes Next

You have now trained a network on real measurements, read its loss curves, checked it species by species and looked at how confident it was when it was wrong. That is a complete piece of machine learning work.

- **Chapter 12, Datasets, DataLoaders and Batching,** feeds the loop small batches instead of all 105 flowers at once, which is what makes large datasets possible.
- **Chapter 13, Optimizers: SGD, Momentum and Adam,** replaces the plain downhill step with the ones used in practice.

Chapter 12 starts with a problem this chapter quietly avoided. All 105 flowers fitted in memory at once, and the photographs in Book 7 will not.